# Food.com Ingredient Network
## 02805 Social Graphs and Interactions — Final Project, DTU Fall 2026

Food.com has been collecting recipes since 2000. The public dump has 231,637 recipes and over a million reviews — essentially a snapshot of almost twenty years of English-language home cooking.

We wanted to know: **if you ignore all the cuisine labels and just look at which ingredients appear together, do natural culinary communities emerge?** And if they do — does each community also produce a distinct *style of writing* when people describe how to cook?

The pipeline: build a co-occurrence graph → detect communities → test them against a null model and check robustness → validate against cuisine tags → check whether users stick to one community → extract technique vocabulary from cooking instructions (ingredient names removed) → check if any of this predicts whether people enjoy the recipe (spoiler: it doesn't).

---
### Setup

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import ast
import random
import re
from collections import Counter, defaultdict
from itertools import combinations
from scipy import stats

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['font.size'] = 10
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import normalized_mutual_info_score
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import community as community_louvain

np.random.seed(42)
random.seed(42)

import os
# Dataset folder: inside the repo, or next to it (e.g. ~/Desktop/food_com_recipes)
DATA = next((d for d in ['food_com_recipes/', '../food_com_recipes/']
             if os.path.exists(os.path.join(d, 'RAW_recipes.csv'))), 'food_com_recipes/')
print('Setup complete')

---
### The data

We filter to recipes with at least 3 reviews (stability) and interactions with rating > 0 (actual ratings, not null-placeholder rows).

In [ ]:
# Load recipes
recipes = pd.read_csv(DATA + 'RAW_recipes.csv')
print(f'Recipes: {len(recipes):,}')

# Parse ingredients from string representation of list
recipes['ingr_list'] = recipes['ingredients'].apply(ast.literal_eval)

# Load interactions, drop 0-ratings (these are unrated placeholders in Food.com)
interactions = pd.read_csv(DATA + 'RAW_interactions.csv')
interactions = interactions[interactions['rating'] > 0].copy()
print(f'Interactions (rating > 0): {len(interactions):,}')
print(f'Rating distribution:\n{interactions["rating"].value_counts().sort_index()}')

In [ ]:
# Compute mean rating per recipe
recipe_ratings = (
    interactions
    .groupby('recipe_id')
    .agg(mean_rating=('rating', 'mean'), n_reviews=('rating', 'count'))
    .reset_index()
)

# Merge ratings onto recipes; keep only recipes with >= 3 reviews for stability
recipes = recipes.merge(recipe_ratings, left_on='id', right_on='recipe_id', how='inner')
recipes = recipes[recipes['n_reviews'] >= 3].copy()
print(f'Recipes with ≥3 reviews: {len(recipes):,}')
print(f'Mean rating overall: {recipes["mean_rating"].mean():.3f}')

In [ ]:
# Identify top-500 ingredients by frequency across all recipes
all_ingr = []
for ingr_list in recipes['ingr_list']:
    all_ingr.extend(ingr_list)
ingr_freq = Counter(all_ingr)

TOP_K = 500
top_ingr = set(ing for ing, _ in ingr_freq.most_common(TOP_K))
print(f'Top-{TOP_K} ingredients cover {sum(ingr_freq[i] for i in top_ingr)/sum(ingr_freq.values()):.1%} of all ingredient mentions')

# Filter each recipe's ingredients to the top-K set
recipes['ingr_filtered'] = recipes['ingr_list'].apply(lambda lst: [i for i in lst if i in top_ingr])

# Keep only recipes that have >= 2 filtered ingredients (need pairs for edges)
recipes = recipes[recipes['ingr_filtered'].apply(len) >= 2].copy()
print(f'Recipes with ≥2 top-K ingredients: {len(recipes):,}')

In [ ]:
# Quick feel for the data before we build anything
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

# Top 20 ingredients
top20 = ingr_freq.most_common(20)
t20_names, t20_counts = zip(*top20)
ax1.barh(t20_names[::-1], t20_counts[::-1], color='#4292c6', alpha=0.85)
ax1.set_xlabel('Appearances across all recipes')
ax1.set_title('Top 20 ingredients')
for sp in ['top','right']: ax1.spines[sp].set_visible(False)

# Rating distribution
rcount = interactions[interactions['rating'] > 0]['rating'].value_counts().sort_index()
ax2.bar(rcount.index, rcount.values,
        color=['#d73027','#fc8d59','#fee08b','#91cf60','#1a9850'][:len(rcount)],
        edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Star rating')
ax2.set_ylabel('Reviews')
pct5 = (interactions['rating'] == 5).mean()
ax2.set_title(f'Rating distribution\n{pct5:.0%} five-star — this matters for VADER later')
for sp in ['top','right']: ax2.spines[sp].set_visible(False)

plt.tight_layout()
plt.show()
def share(ingr):
    return recipes['ingr_list'].apply(lambda lst: ingr in lst).mean()
print(f"Salt appears in {share('salt'):.0%} of recipes.")
print(f"Butter: {share('butter'):.0%}.  Garlic: {share('garlic cloves'):.0%}.")

---
### Building the network

Nodes = ingredients. Edge = co-appear in at least 5 recipes. Edge weight = count. We use the 500 most common ingredients so the network stays tractable and we're looking at meaningful signal (rare ingredients would mostly add noise).

The threshold of 5 is a choice, so further down we check that the communities survive other thresholds (1, 3, 10, 20) and a principled backbone (the disparity filter).

In [ ]:
# Nodes: top-500 ingredients
# Edges: two ingredients are connected if they appear in the same recipe
# Edge weight: number of recipes they co-occur in

edge_counts = Counter()
for ingr_list in recipes['ingr_filtered']:
    for a, b in combinations(sorted(ingr_list), 2):
        edge_counts[(a, b)] += 1

print(f'Total potential edges: {len(edge_counts):,}')
# A recipe that lists the same ingredient twice (e.g. salt in crust and filling) creates a self-pair
n_self = sum(1 for (a, b) in edge_counts if a == b)
print(f'Self-pairs from duplicated ingredients: {n_self}')

def build_graph(threshold):
    """Co-occurrence graph keeping pairs seen in >= threshold recipes (largest component)."""
    H = nx.Graph()
    H.add_nodes_from(sorted(top_ingr))  # sorted: set order changes per session and would change Louvain results
    for (a, b), w in edge_counts.items():
        if w >= threshold:
            H.add_edge(a, b, weight=w)
    lcc = max(nx.connected_components(H), key=len)
    return H.subgraph(lcc).copy()

# Backbone threshold: keep edges with weight >= 5 to remove noise
EDGE_THRESHOLD = 5
G = build_graph(EDGE_THRESHOLD)
lcc = set(G.nodes())

n_removed = len(edge_counts) - G.number_of_edges()
print(f'Network (threshold={EDGE_THRESHOLD}): {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Removed by threshold: {n_removed:,} pairs ({n_removed/len(edge_counts):.0%})')
print(f'Density: {nx.density(G):.4f}')
print(f'Average degree: {sum(d for _,d in G.degree())/G.number_of_nodes():.1f}')

---
### What does the network look like?

Before community detection, check basic structure — degree distribution and betweenness centrality.

In [ ]:
# Degree distribution
degrees = [d for _, d in G.degree()]
print(f'Degree: min={min(degrees)}, max={max(degrees)}, mean={np.mean(degrees):.1f}, median={np.median(degrees):.0f}')

# Top ingredients by (weighted) degree
weighted_degree = dict(G.degree(weight='weight'))
top_by_degree = sorted(weighted_degree.items(), key=lambda x: x[1], reverse=True)[:20]
print('\nTop 10 ingredients by weighted degree:')
for ingr, wd in top_by_degree[:10]:
    print(f'  {ingr}: {wd:,}')

In [ ]:
# Betweenness centrality (sample-based for speed)
# networkx treats `weight` as a path LENGTH, so frequent co-occurrence must be
# converted to a short distance (1/weight). k=200 = number of sampled SOURCE nodes
# (shortest paths are computed from 200 random sources to all targets). Seed fixes the sample.
for a, b, d in G.edges(data=True):
    d['distance'] = 1.0 / d['weight']

print('Computing betweenness centrality (k=200 sampled source nodes)...')
betweenness = nx.betweenness_centrality(G, k=min(200, G.number_of_nodes()), weight='distance', normalized=True, seed=42)

print('Top 10 by betweenness:')
for ingr, bc in sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {ingr}: {bc:.4f}')
print('\nWith 1/weight distances, shortest paths run through the heaviest edges, i.e. through pantry staples.')
print('So betweenness here finds HUBS. For ingredients that genuinely connect communities we use the')
print('participation coefficient further down.')

## Community detection

Louvain on the weighted network. Louvain is stochastic, so we report the partition for seed 42 and check how stable it is across 10 other seeds.

Then the key question: **is the modularity we find more than you'd get from a network with no communities at all?** Following the week 2 logic (degree-preserving shuffles), we compare against a **strength-preserving configuration model**: randomly re-wire the co-occurrences while every ingredient keeps its total co-occurrence count (its strength), run Louvain on each random network, and see where the real Q falls.

In [ ]:
# Louvain community detection — main partition (seed 42)
partition = community_louvain.best_partition(G, weight='weight', random_state=42)
modularity_obs = community_louvain.modularity(partition, G, weight='weight')

community_sizes = Counter(partition.values())
n_communities = len(community_sizes)
print(f'Communities found: {n_communities}')
print(f'Modularity Q (observed): {modularity_obs:.4f}')
print(f'Community sizes: {sorted(community_sizes.values(), reverse=True)}')

# Stability: 10 more seeds. How much does Q vary, and do the partitions agree with the main one?
node_order = list(G.nodes())
def as_labels(p, nodes=node_order):
    return [p[n] for n in nodes]

seed_runs = []
for s in range(10):
    p_s = community_louvain.best_partition(G, weight='weight', random_state=s)
    seed_runs.append({
        'seed': s,
        'Q': community_louvain.modularity(p_s, G, weight='weight'),
        'n_comm': len(set(p_s.values())),
        'NMI_vs_main': normalized_mutual_info_score(as_labels(partition), as_labels(p_s)),
    })
seed_df = pd.DataFrame(seed_runs)
print('\nStability across 10 seeds:')
print(f"  Q range: {seed_df['Q'].min():.4f} – {seed_df['Q'].max():.4f}")
print(f"  Number of communities: {sorted(seed_df['n_comm'].unique())}")
print(f"  NMI vs main partition: mean {seed_df['NMI_vs_main'].mean():.3f}, min {seed_df['NMI_vs_main'].min():.3f}")

In [ ]:
# Null model: strength-preserving configuration model (weighted version of the
# degree-preserving shuffle from week 2).
# An edge of weight w is w co-occurrence "units". Cut every unit into two stubs,
# shuffle all stubs, and pair them up again: each ingredient keeps exactly its
# strength, but WHO it co-occurs with is random. Then let Louvain do its best on
# that random network. If real Q >> null Q, the communities are not just a
# by-product of some ingredients being more common than others.
N_NULL = 100   # ~a few minutes; Louvain is run once per random network
null_nodes = list(G.nodes())
n_nodes = len(null_nodes)
strength = np.array([weighted_degree[n] for n in null_nodes], dtype=np.int64)

def strength_preserving_null(rng):
    stubs = np.repeat(np.arange(n_nodes), strength)
    rng.shuffle(stubs)
    a, b = stubs[0::2], stubs[1::2]
    keep = a != b                      # drop self-pairs (an ingredient with itself)
    lo = np.minimum(a[keep], b[keep])
    hi = np.maximum(a[keep], b[keep])
    counts = np.bincount(lo * n_nodes + hi, minlength=n_nodes * n_nodes).reshape(n_nodes, n_nodes)
    i, j = np.nonzero(counts)
    H = nx.Graph()
    H.add_nodes_from(range(n_nodes))
    H.add_weighted_edges_from(zip(i.tolist(), j.tolist(), counts[i, j].tolist()))
    return H

rng = np.random.default_rng(42)
null_mods = []
print(f'Running strength-preserving null ({N_NULL} random networks)...')
for it in range(N_NULL):
    H = strength_preserving_null(rng)
    p_null = community_louvain.best_partition(H, weight='weight', random_state=it)
    null_mods.append(community_louvain.modularity(p_null, H, weight='weight'))
    if (it + 1) % 25 == 0:
        print(f'  {it + 1}/{N_NULL}')
null_mods = np.array(null_mods)

null_mean = null_mods.mean()
null_std  = null_mods.std()
z_score   = (modularity_obs - null_mean) / null_std
p_null_emp = (1 + (null_mods >= modularity_obs).sum()) / (1 + N_NULL)   # week 2: never report p = 0

print(f'Null Q (Louvain on random networks): {null_mean:.4f} ± {null_std:.4f}  (max {null_mods.max():.4f})')
print(f'Observed Q: {modularity_obs:.4f}')
print(f'z-score: {z_score:.1f},  empirical p = {p_null_emp:.3f} (smallest possible with {N_NULL} samples: {1/(1+N_NULL):.3f})')

# Look at the histogram before quoting z (week 2)
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(null_mods, bins=20, color='#999999', label='Louvain Q on null networks')
ax.axvline(modularity_obs, color='#d62728', lw=2, label=f'Observed Q = {modularity_obs:.3f}')
ax.set_xlabel('Modularity Q'); ax.set_ylabel('Count'); ax.legend(fontsize=8)
ax.set_title('Strength-preserving null model')
plt.tight_layout()
plt.savefig('fig_supp_null.png', dpi=150, bbox_inches='tight')
plt.show()

### Robustness: does the community structure depend on the threshold?

The threshold of 5 is arbitrary, so we rebuild the network at other thresholds, and also with the **disparity filter** backbone (Serrano, Boguñá & Vespignani 2009, week 4), which keeps an edge only if its weight is surprisingly large compared to how that ingredient's strength is spread over its links (α = 0.05). For each version we compare the communities with our main partition (NMI on shared nodes) and check whether the Asian cluster stays together.

In [ ]:
def disparity_backbone(H, alpha=0.05):
    """Disparity filter: keep edge if significant (alpha) for at least one endpoint."""
    s = dict(H.degree(weight='weight'))
    k = dict(H.degree())
    B = nx.Graph()
    B.add_nodes_from(H.nodes())
    for u, v, d in H.edges(data=True):
        w = d['weight']
        a_u = (1 - w / s[u]) ** (k[u] - 1) if k[u] > 1 else 0.0
        a_v = (1 - w / s[v]) ** (k[v] - 1) if k[v] > 1 else 0.0
        if min(a_u, a_v) < alpha:
            B.add_edge(u, v, weight=w)
    lcc_b = max(nx.connected_components(B), key=len)
    return B.subgraph(sorted(lcc_b)).copy()

def best_louvain(H, seeds=range(5)):
    best = None
    for s in seeds:
        p = community_louvain.best_partition(H, weight='weight', random_state=s)
        q = community_louvain.modularity(p, H, weight='weight')
        if best is None or q > best[0]:
            best = (q, p)
    return best

# Which main community is the Asian one? (by content, same rule as the naming cell below)
main_asian = next(c for c in set(partition.values())
                  if {'soy sauce', 'sesame oil'} & {n for n in partition if partition[n] == c})
main_asian_nodes = [n for n in partition if partition[n] == main_asian]

variants = {f'threshold ≥ {t}': build_graph(t) for t in [1, 3, 5, 10, 20]}
variants['disparity filter (α=0.05)'] = disparity_backbone(build_graph(1), alpha=0.05)

robust_rows = []
for name, H in variants.items():
    q, p = best_louvain(H)
    shared = [n for n in H.nodes() if n in partition]
    asian_in_H = [n for n in main_asian_nodes if n in p]
    asian_together = (max(Counter(p[n] for n in asian_in_H).values()) / len(asian_in_H)) if asian_in_H else np.nan
    robust_rows.append({
        'network': name,
        'nodes': H.number_of_nodes(),
        'edges': H.number_of_edges(),
        'density': round(nx.density(H), 3),
        'Q': round(q, 3),
        'communities': len(set(p.values())),
        'NMI_vs_main': round(normalized_mutual_info_score(as_labels(partition, shared), as_labels(p, shared)), 3),
        'Asian_kept_together': round(asian_together, 2),
    })
robust_df = pd.DataFrame(robust_rows)
print(robust_df.to_string(index=False))

In [ ]:
# Label communities by their most characteristic ingredients
# (top 5 by frequency within each community)
community_to_nodes = defaultdict(list)
for node, comm in partition.items():
    community_to_nodes[comm].append(node)

# Manual inspection: print top nodes per community
print('Top nodes per community (sorted by weighted degree):')
comm_labels = {}
for comm_id in sorted(community_sizes, key=community_sizes.get, reverse=True):
    nodes = community_to_nodes[comm_id]
    top_nodes = sorted(nodes, key=lambda n: weighted_degree.get(n, 0), reverse=True)[:8]
    print(f'  Community {comm_id} ({len(nodes)} nodes): {top_nodes}')
    comm_labels[comm_id] = ', '.join(top_nodes[:3])

In [ ]:
# Name communities by content (not by arbitrary Louvain ID which changes between runs)
# Also define consistent colors for all figures
name_to_color = {'Savory': '#1f77b4', 'Baking': '#ff7f0e', 'Asian': '#2ca02c'}

comm_name  = {}   # comm_id → 'Savory' / 'Baking' / 'Asian'
comm_color = {}   # comm_id → hex color

for comm_id, nodes in community_to_nodes.items():
    node_set = set(nodes)
    if 'soy sauce' in node_set or 'sesame oil' in node_set:
        name = 'Asian'
    elif 'sugar' in node_set and 'flour' in node_set and 'baking powder' in node_set:
        name = 'Baking'
    else:
        name = 'Savory'
    comm_name[comm_id]  = name
    comm_color[comm_id] = name_to_color[name]

print('Community labels:')
for comm_id in sorted(comm_name, key=lambda c: community_sizes[c], reverse=True):
    top5 = sorted(community_to_nodes[comm_id],
                  key=lambda n: weighted_degree.get(n, 0), reverse=True)[:5]
    print(f'  {comm_name[comm_id]:7s}  ({community_sizes[comm_id]} nodes): {top5}')

### Which ingredients actually connect communities?

Betweenness (above) is dominated by staples. The **participation coefficient** (Guimerà & Amaral 2005) asks a more direct question: how evenly is an ingredient's co-occurrence weight spread across the communities?
$P_i = 1 - \sum_c (s_{i,c}/s_i)^2$, where $s_{i,c}$ is the weight from $i$ to community $c$. $P = 0$: all links stay in its own community; with three communities the maximum is $2/3$.

In [ ]:
s_to_comm = defaultdict(lambda: defaultdict(float))
for u, v, d in G.edges(data=True):
    s_to_comm[u][partition[v]] += d['weight']
    s_to_comm[v][partition[u]] += d['weight']
participation = {n: 1 - sum((x / weighted_degree[n]) ** 2 for x in s_to_comm[n].values()) for n in G.nodes()}

print('Mean participation coefficient by community:')
for cid, cname in comm_name.items():
    vals = [participation[n] for n in community_to_nodes[cid]]
    print(f'  {cname:7s}: {np.mean(vals):.3f}')

# Restrict to reasonably common ingredients (weighted degree above the median) so
# the list isn't dominated by rare ingredients with a handful of links.
wd_median = np.median(list(weighted_degree.values()))
common = [n for n in G.nodes() if weighted_degree[n] >= wd_median]
top_bridges_P = sorted(common, key=lambda n: participation[n], reverse=True)[:15]
print('\nTop 15 connector ingredients by participation coefficient (weighted degree ≥ median):')
for n in top_bridges_P:
    shares = {comm_name[c]: s_to_comm[n][c] / weighted_degree[n] for c in comm_name}
    share_txt = ', '.join(f'{k} {v:.0%}' for k, v in sorted(shares.items(), key=lambda x: -x[1]))
    print(f'  {n:28s} P={participation[n]:.3f}  [{comm_name[partition[n]]}]  links: {share_txt}')

In [ ]:
# Assign each recipe to its plurality community
# (the community holding the most of its filtered ingredients)
# Using apply() — much faster than iterrows on 87K rows

def majority_comm(ingr_list):
    counts = Counter(partition[i] for i in ingr_list if i in partition)
    return counts.most_common(1)[0][0] if counts else None

recipes['primary_comm'] = recipes['ingr_filtered'].apply(majority_comm)
recipes['comm_name']    = recipes['primary_comm'].map(comm_name)
recipe_primary_comm     = (recipes.dropna(subset=['primary_comm'])
                           .set_index('id')['primary_comm'].astype(int).to_dict())

print(f'Recipes assigned to a community: {recipes["primary_comm"].notna().sum():,}')
print(recipes['comm_name'].value_counts())

In [ ]:
# Make the communities tangible — show actual recipe names from each one
# (filter to recipes with enough reviews so we're not showing obscure ones)
print('Sample recipes from each community (filtered to ≥50 reviews, sorted by rating):\n')
for cname in ['Savory', 'Baking', 'Asian']:
    subset = recipes[(recipes['comm_name'] == cname) & (recipes['n_reviews'] >= 50)]
    if len(subset) == 0:
        continue
    sample = subset.nlargest(200, 'mean_rating').sample(min(6, len(subset)), random_state=42)
    print(f'  {cname}  ({len(recipes[recipes["comm_name"]==cname]):,} recipes total):')
    for _, row in sample.iterrows():
        print(f'    · {row["name"]:45s}  ★{row["mean_rating"]:.2f}  ({row["n_reviews"]} reviews)')
    print()

In [ ]:
# External validation: NMI between network communities and Food.com cuisine tags
recipes['tags_list'] = recipes['tags'].apply(ast.literal_eval)

asian_tags  = {'asian','chinese','japanese','korean','thai','vietnamese'}
baking_tags = {'baking','cake','cakes','cookie','cookies','dessert','desserts',
               'pastry','muffin','muffins','pie','pies','bread'}
savory_tags = {'main-dish','chicken','beef','pasta','soup','stew','meat','pork','seafood'}

def cuisine_label(tags):
    t = {x.lower() for x in tags}
    if t & asian_tags:  return 'Asian'
    if t & baking_tags: return 'Baking'
    if t & savory_tags: return 'Savory'
    return None

recipes['cuisine_label'] = recipes['tags_list'].apply(cuisine_label)
labeled = recipes[recipes['cuisine_label'].notna() & recipes['comm_name'].notna()].copy()
print(f'Recipes with both network community and cuisine tag: {len(labeled):,}')

nmi = normalized_mutual_info_score(labeled['cuisine_label'], labeled['comm_name'])

# Permutation null (200 shuffles)
cl = labeled['cuisine_label'].to_numpy(dtype=object)
cn = labeled['comm_name'].to_numpy(dtype=object)
null_nmis = []
for _ in range(200):
    null_nmis.append(normalized_mutual_info_score(np.random.permutation(cl), cn))

print(f'\nNMI (network community vs. cuisine tag): {nmi:.3f}')
print(f'Null NMI: {np.mean(null_nmis):.2e} ± {np.std(null_nmis):.2e}')
print(pd.crosstab(labeled['comm_name'], labeled['cuisine_label']))
print('The network communities align reasonably with cuisine tags —')
print('imperfect because many recipes span traditions.')

## Does community membership predict recipe quality?

Spoiler: no. But it's worth running the ANOVA properly.

In [ ]:
# For each ingredient, compute its mean rating across all recipes it appears in
ingr_ratings = defaultdict(list)
for ingr_list, rating in zip(recipes['ingr_filtered'], recipes['mean_rating']):
    for ingr in ingr_list:
        if ingr in G:
            ingr_ratings[ingr].append(rating)

ingr_mean_rating = {ingr: np.mean(ratings) for ingr, ratings in ingr_ratings.items() if len(ratings) >= 20}
print(f'Ingredients with ≥20 recipe ratings: {len(ingr_mean_rating)}')

# Build a dataframe with network metrics + rating per ingredient
ingr_df = pd.DataFrame([
    {
        'ingredient': ingr,
        'degree': G.degree(ingr),
        'weighted_degree': weighted_degree.get(ingr, 0),
        'betweenness': betweenness.get(ingr, 0),
        'participation': participation.get(ingr, 0),
        'community': partition.get(ingr, -1),
        'mean_rating': ingr_mean_rating[ingr],
        'n_recipes': len(ingr_ratings[ingr])
    }
    for ingr in ingr_mean_rating
    if ingr in G.nodes() and ingr in partition
])

print(ingr_df.describe())

In [ ]:
# Correlation: degree vs. mean rating
r_deg, p_deg = stats.pearsonr(ingr_df['degree'], ingr_df['mean_rating'])
r_bet, p_bet = stats.pearsonr(ingr_df['betweenness'], ingr_df['mean_rating'])
print(f'Degree vs. mean rating:      r={r_deg:.3f}, p={p_deg:.3e}')
print(f'Betweenness vs. mean rating: r={r_bet:.3f}, p={p_bet:.3e}')

# Null check: shuffle ratings 1000 times
null_rs = []
ratings_arr = ingr_df['mean_rating'].values.copy()
degrees_arr = ingr_df['degree'].values
for _ in range(1000):
    shuffled = ratings_arr.copy()
    np.random.shuffle(shuffled)
    null_rs.append(stats.pearsonr(degrees_arr, shuffled)[0])

null_r_mean = np.mean(null_rs)
null_r_std = np.std(null_rs)
z_rating = (r_deg - null_r_mean) / null_r_std
print(f'\nNull (shuffled ratings): r={null_r_mean:.4f} ± {null_r_std:.4f}')
print(f'Observed r={r_deg:.3f}, z={z_rating:.1f}')

In [ ]:
# One-way ANOVA: do communities produce differently-rated recipes?
# Using plurality community assignment (one recipe = one community)
assert 'comm_name' in recipes.columns, 'Run the community-assignment cell (recipes["comm_name"]) first'
print('Mean recipe rating by community (plurality assignment):')
groups = {}
for cname in ['Savory', 'Baking', 'Asian']:
    r = recipes[recipes['comm_name'] == cname]['mean_rating'].values
    groups[cname] = r
    print(f'  {cname:7s}: {np.mean(r):.3f} ± {np.std(r):.3f}  (n={len(r):,})')

f_stat, p_anova = stats.f_oneway(*groups.values())
# Effect size: share of rating variance explained by community (eta squared)
all_r = np.concatenate(list(groups.values()))
ss_between = sum(len(g) * (g.mean() - all_r.mean())**2 for g in groups.values())
eta_sq = ss_between / ((all_r - all_r.mean())**2).sum()
print(f'\nANOVA: F={f_stat:.2f}, p={p_anova:.3f}, eta²={eta_sq:.5f}')
if p_anova > 0.05:
    print('→ No detectable difference (and the effect size is negligible either way)')
else:
    print(f'→ Statistically significant only because n is huge; community explains '
          f'{eta_sq:.3%} of rating variance — practically no effect')

## Do cooks stay in one community?

So far the network is made of ingredients. But Food.com is also a **social** system: users review recipes. That gives a bipartite **user–recipe network** (an edge = a review), and each recipe belongs to an ingredient community. The question: **do users concentrate their reviews in one culinary community more than chance would predict?**

- For each active user (≥ 10 reviews of recipes we could assign), compute the share of their reviews in each community and the **concentration** $H_u = \sum_c \text{share}_{u,c}^2$ (1 = only one community; 1/3 = perfectly even).
- **Null model:** shuffle which community each review points to, across all reviews. Every user keeps their number of reviews and every community keeps its total number of reviews (degrees are preserved on both sides of the bipartite network) — only the *matching* is random. Savory is much bigger than Asian, so this null, not 1/3, is the right baseline.
- Then a link back to ratings: does a user rate recipes from their **home** community (where most of their reviews are) higher than recipes from elsewhere?

In [ ]:
ui = interactions[['user_id', 'recipe_id', 'rating']].copy()
ui['comm'] = ui['recipe_id'].map(recipe_primary_comm)
ui = ui.dropna(subset=['comm'])
ui['comm'] = ui['comm'].astype(int)

MIN_REVIEWS = 10
user_counts = ui['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_REVIEWS].index
ua = ui[ui['user_id'].isin(active_users)].copy()
print(f'User–recipe network: {ui["user_id"].nunique():,} users, {ui["recipe_id"].nunique():,} recipes, {len(ui):,} reviews')
print(f'Active users (≥{MIN_REVIEWS} reviews): {len(active_users):,}, with {len(ua):,} reviews '
      f'({len(ua)/len(ui):.0%} of all reviews)')

comm_ids   = sorted(comm_name)                       # fixed order of communities
comm_pos   = {c: i for i, c in enumerate(comm_ids)}
user_codes, user_index = pd.factorize(ua['user_id'])
comm_idx   = ua['comm'].map(comm_pos).to_numpy()
n_users    = len(user_index)

def user_shares(cidx):
    tab = np.zeros((n_users, len(comm_ids)))
    np.add.at(tab, (user_codes, cidx), 1)
    return tab / tab.sum(axis=1, keepdims=True)

shares_obs = user_shares(comm_idx)
H_obs = (shares_obs ** 2).sum(axis=1)
home_share_obs = shares_obs.max(axis=1)

# Null: shuffle the community of each review (degree-preserving on both sides)
rng_u = np.random.default_rng(42)
N_USER_NULL = 200
H_null_means, home_null_means = [], []
SPEC = 0.8                                           # 'specialist': ≥80% of reviews in one community
spec_null = {c: [] for c in comm_ids}
for _ in range(N_USER_NULL):
    sh = user_shares(rng_u.permutation(comm_idx))
    H_null_means.append((sh ** 2).sum(axis=1).mean())
    home_null_means.append(sh.max(axis=1).mean())
    for c in comm_ids:
        spec_null[c].append((sh[:, comm_pos[c]] >= SPEC).mean())
H_null_means = np.array(H_null_means)

z_users = (H_obs.mean() - H_null_means.mean()) / H_null_means.std()
p_users = (1 + (H_null_means >= H_obs.mean()).sum()) / (1 + N_USER_NULL)
print(f'\nMean concentration H: observed {H_obs.mean():.3f}  vs null {H_null_means.mean():.3f} ± {H_null_means.std():.4f}')
print(f'z = {z_users:.1f}, empirical p = {p_users:.3f}')
print(f'Share of reviews in the home community: observed {home_share_obs.mean():.1%} vs null {np.mean(home_null_means):.1%}')

print(f'\nSpecialists (≥{SPEC:.0%} of reviews in one community), observed vs null:')
specialists = {}
for c in comm_ids:
    obs = (shares_obs[:, comm_pos[c]] >= SPEC).mean()
    specialists[comm_name[c]] = (obs, np.mean(spec_null[c]))
    print(f'  {comm_name[c]:7s}: {obs:.1%} of users  vs {np.mean(spec_null[c]):.1%} expected')

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(H_obs, bins=30, alpha=0.7, color='#1f77b4', label='observed users')
ax.axvline(H_null_means.mean(), color='#d62728', lw=2, label='null mean')
ax.set_xlabel('Concentration H of a user\'s reviews'); ax.set_ylabel('Users'); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('fig_supp_users.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Home vs away: do users rate recipes from their home community higher?
# (home = the community with most of the user's reviews; paired comparison within users)
home_of = dict(zip(user_index, np.array(comm_ids)[shares_obs.argmax(axis=1)]))
ua['at_home'] = ua['comm'].to_numpy() == ua['user_id'].map(home_of).to_numpy()
per_user = (ua.groupby(['user_id', 'at_home'])['rating'].agg(['mean', 'count'])
              .unstack('at_home'))
per_user = per_user[(per_user[('count', True)] >= 3) & (per_user[('count', False)] >= 3)]
diff_home = (per_user[('mean', True)] - per_user[('mean', False)]).to_numpy()
w_stat, p_home = stats.wilcoxon(diff_home)
print(f'Users with ≥3 home and ≥3 away reviews: {len(diff_home):,}')
print(f'Mean rating home − away: {diff_home.mean():+.3f} stars (median {np.median(diff_home):+.3f}); '
      f'Wilcoxon p = {p_home:.2e}')
print(f'Users who rate home higher: {(diff_home > 0).mean():.0%}, lower: {(diff_home < 0).mean():.0%}')

## Cooking language by community

Reviews are useless here — 76% are five-star and they all say something like "made this for dinner, everyone loved it!" The **cooking instructions** are where community character shows up.

**Careful — this could be circular.** The communities were built from ingredients, and recipe steps name those ingredients ("add the soy sauce", "sift the flour"). Finding *soy* in Asian steps would just re-detect the ingredients. So we **remove every top-500 ingredient name from the step text first**. Whatever difference is left is about *how* people cook (technique), not *what* they cook with.

We treat **each recipe as one document**, fit TF-IDF over all sampled recipes, and average the TF-IDF vectors within each community. (Putting each community into one giant document would leave only 3 documents, so IDF could only take 3 values and a word would score high just for being absent from one community.) A word is *distinctive* for a community when its average score there is higher than its average in the other two.

To check that the difference is real and not just a few cherry-picked words, we train a classifier to guess a recipe's community from its (ingredient-free) step text, on a balanced sample, and compare its cross-validated accuracy with the same classifier trained on shuffled labels.

In [ ]:
# Parse cooking step text (stored as a string representation of a list)
print('Parsing cooking steps...')
recipes['steps_list'] = recipes['steps'].apply(ast.literal_eval)
recipes['steps_text'] = recipes['steps_list'].apply(lambda s: ' '.join(s).lower())

# Sample up to 3000 recipes per community; every recipe is its own document
SAMPLE_N = 3000
corpus_names = ['Savory', 'Baking', 'Asian']
docs, doc_comm = [], []

for cname in corpus_names:
    subset = recipes[recipes['comm_name'] == cname]
    sample  = subset.sample(min(SAMPLE_N, len(subset)), random_state=42)
    docs.extend(sample['steps_text'].tolist())
    doc_comm.extend([cname] * len(sample))
    print(f'  {cname:7s}: {len(sample):,} recipes')
doc_comm = np.array(doc_comm)

# --- Remove ingredient names from the step text ---
# Ingredient names are matched as whole phrases (longest first, so 'soy sauce' goes
# before 'sauce' could), with simple singular/plural variants ('eggs' ~ 'egg').
def tokens(text):
    return re.findall(r'[a-z]+', text.lower())

def name_variants(phrase):
    v = {phrase, phrase + 's', phrase + 'es'}
    if phrase.endswith('es'): v.add(phrase[:-2])
    if phrase.endswith('s'):  v.add(phrase[:-1])
    return v

ingr_phrases = set()
for ing in top_ingr:
    ingr_phrases |= name_variants(' '.join(tokens(ing)))
max_len = max(len(p.split()) for p in ingr_phrases)

def mask_ingredients(text):
    toks, out, i = tokens(text), [], 0
    while i < len(toks):
        for n in range(min(max_len, len(toks) - i), 0, -1):
            if ' '.join(toks[i:i + n]) in ingr_phrases:
                i += n
                break
        else:
            out.append(toks[i]); i += 1
    return ' '.join(out)

docs_raw = docs
docs = [mask_ingredients(d) for d in docs_raw]

# Stricter version for a sensitivity check: drop EVERY word that occurs in any
# ingredient name (so 'soy' goes even when the text just says 'soy', but technique
# words that are also part of a name, like 'cream' or 'ground', are lost too)
ingr_words = {w for p in ingr_phrases for w in p.split()}
docs_strict = [' '.join(t for t in d.split() if t not in ingr_words) for d in docs]
n_before = sum(len(tokens(d)) for d in docs_raw)
n_after  = sum(len(d.split()) for d in docs)
print(f'\nIngredient names removed: {1 - n_after / n_before:.0%} of all step words')
print('Example — before:', docs_raw[0][:160])
print('Example — after: ', docs[0][:160])

In [ ]:
# Generic cooking verbs, units and utensils that appear in every community and
# tell us nothing. They are removed BEFORE fitting (together with English stop words).
COOKING_STOPWORDS = {
    'add','stir','mix','heat','cook','place','remove','pour','combine',
    'set','cut','use','let','put','make','turn','cover','serve',
    'minutes','minute','cup','cups','tablespoon','tablespoons',
    'teaspoon','teaspoons','large','small','medium','well','water',
    'until','ingredients','bowl','pan','oven','time','hour'
}

def make_vectorizer():
    return TfidfVectorizer(
        stop_words=sorted(ENGLISH_STOP_WORDS | COOKING_STOPWORDS),
        max_features=8000,
        ngram_range=(1, 1),
        token_pattern=r'\b[a-z][a-z]+\b',
        min_df=5,            # ignore words used in fewer than 5 recipes (typos, one-offs)
        sublinear_tf=True,   # 1 + log(tf): a word repeated 10 times in one recipe doesn't dominate
    )

TOP_N = 12
def distinctive_words(texts):
    """Mean per-recipe TF-IDF per community, and words ranked by (own mean − mean of the other two)."""
    vec = make_vectorizer()
    X = vec.fit_transform(texts)
    names = vec.get_feature_names_out()
    profile = np.vstack([np.asarray(X[doc_comm == c].mean(axis=0)).ravel() for c in corpus_names])
    top = {}
    for i, cname in enumerate(corpus_names):
        distinct = profile[i] - np.delete(profile, i, axis=0).mean(axis=0)
        idx = distinct.argsort()[::-1][:TOP_N]
        top[cname] = [(names[j], distinct[j]) for j in idx]
    return vec, X, profile, top

# Main analysis: ingredient names removed (technique vocabulary)
vectorizer, X, tfidf_matrix, community_top_words = distinctive_words(docs)
print(f'Document-term matrix: {X.shape[0]:,} recipes × {X.shape[1]:,} words')
print('\nMost distinctive cooking-step words per community (ingredient names REMOVED):\n')
for cname, words in community_top_words.items():
    print(f'  {cname:7s}:  {" | ".join(w for w, _ in words[:8])}')

# For comparison: the same analysis with ingredient names left in
_, _, _, top_words_unmasked = distinctive_words(docs_raw)
print('\nFor comparison, WITH ingredient names (partly circular):\n')
for cname, words in top_words_unmasked.items():
    print(f'  {cname:7s}:  {" | ".join(w for w, _ in words[:8])}')

In [ ]:
# A cleaner look at the vocabulary split (bar length ∝ distinctiveness)
max_s = max(s for words in community_top_words.values() for _, s in words)
for cname, words in community_top_words.items():
    print(f'{cname}')
    for w, s in words[:10]:
        bar = '█' * max(1, int(50 * s / max_s))
        print(f'  {w:20s}  {bar}')
    print()

In [ ]:
# Is the technique difference real? Predict a recipe's community from its
# ingredient-free step text. Balanced classes (chance = 1/3), 5-fold CV, macro-F1.
# Null: the same pipeline with community labels shuffled (20 times).
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

rng_clf = np.random.default_rng(42)
n_bal = min(Counter(doc_comm).values())
bal_idx = np.concatenate([rng_clf.choice(np.where(doc_comm == c)[0], n_bal, replace=False)
                          for c in corpus_names])
docs_bal = [docs[i] for i in bal_idx]
docs_bal_raw = [docs_raw[i] for i in bal_idx]
docs_bal_strict = [docs_strict[i] for i in bal_idx]
y_bal = doc_comm[bal_idx]

def cv_f1(texts, labels):
    pipe = make_pipeline(make_vectorizer(), LogisticRegression(max_iter=2000))
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(pipe, texts, labels, cv=cv, scoring='f1_macro').mean()

f1_masked = cv_f1(docs_bal, y_bal)
f1_raw    = cv_f1(docs_bal_raw, y_bal)
f1_strict = cv_f1(docs_bal_strict, y_bal)
N_CLF_NULL = 20
f1_null = np.array([cv_f1(docs_bal, rng_clf.permutation(y_bal)) for _ in range(N_CLF_NULL)])
p_clf = (1 + (f1_null >= f1_masked).sum()) / (1 + N_CLF_NULL)

print(f'Balanced sample: {n_bal:,} recipes per community')
print(f'Macro-F1, ingredient names removed: {f1_masked:.3f}')
print(f'Macro-F1, strict (every ingredient word removed): {f1_strict:.3f}')
print(f'Macro-F1, ingredient names kept:    {f1_raw:.3f}  (upper reference)')

_, _, _, top_words_strict = distinctive_words(docs_strict)
print('\nDistinctive words with the strict mask:')
for cname, words in top_words_strict.items():
    print(f'  {cname:7s}:  {" | ".join(w for w, _ in words[:8])}')
print(f'Macro-F1, shuffled labels:          {f1_null.mean():.3f} ± {f1_null.std():.3f}  (chance ≈ 0.333)')
print(f'Empirical p = {p_clf:.3f}')

In [ ]:
# Words are abstract — here are actual cooking instructions from each community
# This shows concretely what the vocabulary difference looks like in real text
print('Real recipe steps from each community:\n')
seeds = {'Baking': 7, 'Savory': 13, 'Asian': 3}
for cname in ['Baking', 'Savory', 'Asian']:
    subset = recipes[(recipes['comm_name'] == cname) & (recipes['n_reviews'] >= 30)]
    if len(subset) == 0:
        continue
    recipe  = subset.sample(1, random_state=seeds[cname]).iloc[0]
    steps   = recipe['steps_list']
    step    = steps[len(steps) // 2]  # middle step — usually the most informative
    preview = step[:280] + ('…' if len(step) > 280 else '')
    print(f'[{cname}]  "{recipe["name"]}"  ★{recipe["mean_rating"]:.1f}')
    print(f'  Step {len(steps)//2 + 1} of {len(steps)}: "{preview}"')
    print()

## Sentiment analysis (VADER) + validation

VADER was designed for Twitter; recipe reviews are uniformly glowing ("loved it!!") and not quite the same distribution. We run it anyway but validate against star ratings first to know how much to trust it.

In [ ]:
# Build interactions_filtered: reviews with actual text, assigned to a community
interactions_filtered = interactions[
    interactions['review'].notna() & (interactions['review'].str.len() > 20)
].copy()
interactions_filtered['community'] = interactions_filtered['recipe_id'].map(recipe_primary_comm)
interactions_filtered = interactions_filtered.dropna(subset=['community'])
interactions_filtered['community'] = interactions_filtered['community'].astype(int)
print(f'Reviews with community assignment: {len(interactions_filtered):,}')

# VADER on 5000 sampled reviews
analyzer = SentimentIntensityAnalyzer()
sample_reviews = interactions_filtered.sample(min(5000, len(interactions_filtered)), random_state=42).copy()
sample_reviews['compound'] = sample_reviews['review'].apply(
    lambda text: analyzer.polarity_scores(str(text))['compound']
)
print('\nMean VADER compound by community:')
for cid, cname in comm_name.items():
    s = sample_reviews[sample_reviews['community'] == cid]['compound']
    print(f'  {cname}: {s.mean():.3f} ± {s.std():.3f}  (n={len(s)})')

In [ ]:
# VALIDATION: hand-label 50 review sentences for sentiment
# VADER labels: compound >= 0.05 = positive, <= -0.05 = negative, else neutral
# We label a sample and compute Cohen's kappa

# Draw a stratified sample: 25 high-compound, 25 low-compound
sample_reviews_sorted = sample_reviews.sort_values('compound')
validation_sample = pd.concat([
    sample_reviews_sorted.head(25),   # most negative by VADER
    sample_reviews_sorted.tail(25)    # most positive by VADER
])[['review', 'compound', 'rating']].reset_index(drop=True)

validation_sample['vader_label'] = validation_sample['compound'].apply(
    lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
)

print('Validation sample (50 reviews for hand labeling):')
print(validation_sample[['review', 'compound', 'vader_label']].head(10).to_string())
print('\n... (save this to validation_sample.csv for hand labeling)')
validation_sample.to_csv('validation_sample.csv', index=False)

In [ ]:
# VADER agrees with star ratings (a proxy for ground truth)
# 4-5 stars = positive, 1-2 stars = negative, 3 = neutral
sample_reviews['rating_label'] = sample_reviews['rating'].apply(
    lambda r: 'positive' if r >= 4 else ('negative' if r <= 2 else 'neutral')
)
sample_reviews['vader_label'] = sample_reviews['compound'].apply(
    lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
)

from sklearn.metrics import cohen_kappa_score, classification_report
# Restrict to non-neutral for a cleaner binary check
binary = sample_reviews[sample_reviews['rating_label'] != 'neutral'].copy()
binary = binary[binary['vader_label'] != 'neutral'].copy()
kappa = cohen_kappa_score(binary['rating_label'], binary['vader_label'])
print(f'Cohen kappa (VADER vs. star rating, binary): {kappa:.3f}  on {len(binary):,} of {len(sample_reviews):,} sampled reviews')
print(f"  dropped: {(sample_reviews['rating_label'] == 'neutral').sum()} three-star, "
      f"{((sample_reviews['rating_label'] != 'neutral') & (sample_reviews['vader_label'] == 'neutral')).sum()} VADER-neutral")
print(classification_report(binary['rating_label'], binary['vader_label']))

print('\nNote: star rating is used as a proxy for ground truth.')
print('For the paper, also hand-label the 50 sentences saved to validation_sample.csv')

## 8. Figures for the paper

In [ ]:
# --- FIGURE 1: Ingredient co-occurrence network with communities ---
PLOT_N = 120
top_nodes = sorted(G.nodes(), key=lambda n: G.degree(n, weight='weight'), reverse=True)[:PLOT_N]
Gsub = G.subgraph(top_nodes).copy()

node_colors = [comm_color.get(partition.get(n, -1), '#aaaaaa') for n in Gsub.nodes()]
node_sizes  = [6 + 0.0004 * weighted_degree.get(n, 0) for n in Gsub.nodes()]

pos = nx.spring_layout(Gsub, weight='weight', seed=42, k=0.5)

fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_edges(Gsub, pos, alpha=0.12, width=0.5, edge_color='#999999', ax=ax)
nx.draw_networkx_nodes(Gsub, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)

# Label top-20 nodes
top20 = sorted(Gsub.nodes(), key=lambda n: G.degree(n, weight='weight'), reverse=True)[:20]
nx.draw_networkx_labels(Gsub, pos, {n: n for n in top20}, font_size=6, font_weight='bold', ax=ax)

legend_patches = [
    mpatches.Patch(color=name_to_color[name], label=f'{name} ({community_sizes[cid]} nodes)')
    for cid, name in comm_name.items()
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=8)
ax.set_title(f'Ingredient co-occurrence network — top {PLOT_N} nodes\n'
             f'Q = {modularity_obs:.3f}  vs. {null_mean:.3f} ± {null_std:.3f} in a strength-preserving null', fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig('fig1_network.pdf', dpi=150, bbox_inches='tight')
plt.savefig('fig1_network.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig1_network.pdf')

In [ ]:
# --- FIGURE 2: TF-IDF vocabulary heatmap (cooking step text) ---
# Collect top words across all communities
all_top = []
for words in community_top_words.values():
    all_top.extend([w for w, _ in words[:10]])
seen, top_union = set(), []
for w in all_top:
    if w not in seen:
        seen.add(w); top_union.append(w)
top_union = top_union[:25]

# Build heatmap matrix
heat = np.zeros((len(corpus_names), len(top_union)))
for i, cname in enumerate(corpus_names):
    for j, word in enumerate(top_union):
        idx = vectorizer.vocabulary_.get(word)
        if idx is not None:
            heat[i, j] = tfidf_matrix[i, idx]

heat_norm = heat / (heat.max(axis=1, keepdims=True) + 1e-9)

fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.imshow(heat_norm, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(len(top_union)))
ax.set_xticklabels(top_union, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(len(corpus_names)))
ax.set_yticklabels(corpus_names, fontsize=10)
ax.set_title('Cooking vocabulary by community — mean per-recipe TF-IDF of step text\n'
             '(10 most distinctive words per community; ingredient names and generic cooking words removed; rows normalized to their max)', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8, label='Normalized mean TF-IDF')
plt.tight_layout()
plt.savefig('fig2_tfidf.pdf', dpi=150, bbox_inches='tight')
plt.savefig('fig2_tfidf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig2_tfidf.pdf')

In [ ]:
# --- FIGURE 3: Betweenness centrality — distribution by community + top bridge ingredients ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# (A) Violin plot: betweenness distribution by community
ax = axes[0]
comm_bc = {comm_name[cid]: [betweenness[n]*1000 for n in nodes if n in betweenness]
           for cid, nodes in community_to_nodes.items()}
order = ['Savory', 'Baking', 'Asian']
parts = ax.violinplot([comm_bc[n] for n in order], positions=range(3),
                      showmedians=True, showextrema=True)
for pc, name in zip(parts['bodies'], order):
    pc.set_facecolor(name_to_color[name]); pc.set_alpha(0.7)
ax.set_xticks(range(3)); ax.set_xticklabels(order)
ax.set_ylabel('Betweenness centrality (×10⁻³)')
ax.set_title('A. Distribution by community')

# (B) Top 15 bridge ingredients colored by community
ax = axes[1]
top15 = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:15]
names15  = [n for n, _ in top15]
vals15   = [v*1000 for _, v in top15]
colors15 = [comm_color.get(partition.get(n, -1), '#aaaaaa') for n in names15]
ax.barh(range(len(names15)), vals15[::-1], color=colors15[::-1])
ax.set_yticks(range(len(names15)))
ax.set_yticklabels(names15[::-1], fontsize=8)
ax.set_xlabel('Betweenness centrality (×10⁻³)')
ax.set_title('B. Top 15 ingredients by betweenness')
legend_patches = [mpatches.Patch(color=name_to_color[n], label=n) for n in order]
ax.legend(handles=legend_patches, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('fig3_betweenness.pdf', dpi=150, bbox_inches='tight')
plt.savefig('fig3_betweenness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig3_betweenness.pdf')

In [ ]:
# Bridge ingredient deep-dive
# Which ingredient spans the most shortest paths between communities?
top_bridge = max(betweenness, key=betweenness.get)
top_bridge_comm = comm_name.get(partition.get(top_bridge), '?')
n_appearances = sum(top_bridge in row for row in recipes['ingr_filtered'])

print(f'Top bridge ingredient:  {top_bridge}')
print(f'  Community:            {top_bridge_comm}')
print(f'  Betweenness:          {betweenness[top_bridge]*1000:.2f} × 10⁻³')
print(f'  Appears in:           {n_appearances:,} recipes\n')

print(f'What recipes use {top_bridge} across different culinary traditions?\n')
for cname in ['Baking', 'Savory', 'Asian']:
    has_it = recipes[
        (recipes['comm_name'] == cname) &
        recipes['ingr_filtered'].apply(lambda lst: top_bridge in lst)
    ]
    if len(has_it) == 0:
        print(f'  {cname}: no recipes with {top_bridge}')
        continue
    top3 = has_it.nlargest(3, 'mean_rating')
    print(f'  {cname} ({len(has_it):,} recipes containing {top_bridge}):')
    for _, r in top3.iterrows():
        print(f'    · {r["name"]:50s}  ★{r["mean_rating"]:.1f}')
    print()

# Also find the best Asian bridge (ingredient from Asian community with high betweenness)
asian_comm_id = [cid for cid, name in comm_name.items() if name == 'Asian'][0]
asian_bridge = max(
    (n for n in community_to_nodes[asian_comm_id] if n in betweenness),
    key=lambda n: betweenness[n]
)
print(f'Top Asian-community node by betweenness: {asian_bridge}  ({betweenness[asian_bridge]*1000:.2f} × 10⁻³)')

---
### Numbers for the paper

Everything the paper quotes is printed here, so after a re-run the paper can be checked against this one cell.

In [ ]:
print('=== PAPER STATISTICS ===')
print(f'Dataset: {len(recipes):,} recipes (≥3 rated reviews, ≥2 top-{TOP_K} ingredients); '
      f'{len(interactions):,} rated reviews in total')
print(f'Ingredient network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges '
      f'(threshold {EDGE_THRESHOLD} removed {n_removed:,} of {len(edge_counts):,} pairs)')
print(f'Network density: {nx.density(G):.4f}')
print(f'Largest connected component: {len(lcc)} / {TOP_K} nodes')
print(f'Communities (Louvain, seed 42): {n_communities}, Q={modularity_obs:.4f}, '
      f'sizes { {comm_name[c]: community_sizes[c] for c in comm_name} }')
print(f"Seed stability (10 seeds): Q {seed_df['Q'].min():.4f}–{seed_df['Q'].max():.4f}, "
      f"communities {sorted(seed_df['n_comm'].unique())}, NMI vs main mean {seed_df['NMI_vs_main'].mean():.3f} "
      f"(min {seed_df['NMI_vs_main'].min():.3f})")
print(f'Strength-preserving null: Q = {null_mean:.4f} ± {null_std:.4f} (max {null_mods.max():.4f}), '
      f'z = {z_score:.1f}, p = {p_null_emp:.3f} ({N_NULL} samples)')
print('Robustness:')
print(robust_df.to_string(index=False))
print(f'NMI vs cuisine tags: {nmi:.3f} (null {np.mean(null_nmis):.1e}) on {len(labeled):,} recipes')
print('Recipe ratings by community: ' + ', '.join(f'{k} {v.mean():.3f}±{v.std():.3f} (n={len(v):,})' for k, v in groups.items()))
print(f'ANOVA: F={f_stat:.2f}, p={p_anova:.3f}, eta²={eta_sq:.5f}')
print(f'Degree–rating correlation: r={r_deg:.3f}, p={p_deg:.2e}')
print(f'Null r: {null_r_mean:.4f} ± {null_r_std:.4f}, z={z_rating:.1f}')
print(f'Betweenness–rating correlation: r={r_bet:.3f}, p={p_bet:.2e}')
print('Top 5 betweenness: ' + ', '.join(f'{n} {b:.3f} [{comm_name[partition[n]]}]'
      for n, b in sorted(betweenness.items(), key=lambda x: -x[1])[:5]))
print('Top 5 participation: ' + ', '.join(f'{n} {participation[n]:.3f} [{comm_name[partition[n]]}]'
      for n in top_bridges_P[:5]))
print('Mean participation by community: ' + ', '.join(
      f'{comm_name[c]} {np.mean([participation[n] for n in community_to_nodes[c]]):.3f}' for c in comm_name))
print(f'TF-IDF distinctive words (ingredient names removed, {1 - n_after / n_before:.0%} of step words):')
for cname, words in community_top_words.items():
    print(f'  {cname}: ' + ', '.join(w for w, _ in words[:8]))
print(f'Classifier macro-F1 (balanced, {n_bal:,}/community): masked {f1_masked:.3f}, '
      f'strict mask {f1_strict:.3f}, with ingredients {f1_raw:.3f}, shuffled {f1_null.mean():.3f} ± {f1_null.std():.3f}, p = {p_clf:.3f}')
print(f'User–recipe network: {ui["user_id"].nunique():,} users; {len(active_users):,} active (≥{MIN_REVIEWS} reviews)')
print(f'User concentration H: observed {H_obs.mean():.3f} vs null {H_null_means.mean():.3f} ± {H_null_means.std():.4f}, '
      f'z = {z_users:.1f}, p = {p_users:.3f}; home share {home_share_obs.mean():.1%} vs {np.mean(home_null_means):.1%}')
print('Specialists (≥80% in one community), observed vs null: ' +
      ', '.join(f'{k} {o:.1%} vs {e:.1%}' for k, (o, e) in specialists.items()))
print(f'Home − away rating: {diff_home.mean():+.3f} stars, Wilcoxon p = {p_home:.2e} (n = {len(diff_home):,} users)')
print(f'VADER validation (vs. star ratings): kappa={kappa:.3f} on {len(binary):,} reviews')

In [ ]:
# Sentiment by community
print('\n=== SENTIMENT BY COMMUNITY ===')
sentiment_groups = {}
for cid, cname in comm_name.items():
    revs = interactions_filtered[interactions_filtered['community'] == cid]['review'].dropna()
    sample = revs.sample(min(500, len(revs)), random_state=42)
    scores = [analyzer.polarity_scores(str(r))['compound'] for r in sample]
    sentiment_groups[cname] = scores
    print(f'  {cname}: mean={np.mean(scores):.3f} ± {np.std(scores):.3f}  (n={len(scores)})')

f_sent, p_sent = stats.f_oneway(*sentiment_groups.values())
print(f'  ANOVA F={f_sent:.2f}, p={p_sent:.3e}')

In [ ]:
# Top/bottom ingredients by mean rating (for discussion)
top_rated = ingr_df.nlargest(10, 'mean_rating')[['ingredient', 'mean_rating', 'degree', 'n_recipes']]
bot_rated = ingr_df.nsmallest(10, 'mean_rating')[['ingredient', 'mean_rating', 'degree', 'n_recipes']]
print('Top 10 ingredients by mean rating:')
print(top_rated.to_string(index=False))
print('\nBottom 10 ingredients by mean rating:')
print(bot_rated.to_string(index=False))

---
### Degree distribution

Sanity check — the degree distribution is notably **not** a power law (the network is too dense for that). Most ingredients are connected to >100 others because pantry staples like salt and butter appear everywhere.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
deg_seq = sorted(degrees, reverse=True)
ax.loglog(range(1, len(deg_seq)+1), deg_seq, 'o', ms=3, alpha=0.6, color='#1f77b4')
ax.set_xlabel('Rank')
ax.set_ylabel('Degree')
ax.set_title('Degree distribution (log-log)')
plt.tight_layout()
plt.savefig('fig_supp_degree.pdf', dpi=150, bbox_inches='tight')
plt.savefig('fig_supp_degree.png', dpi=150, bbox_inches='tight')
plt.show()